# Legal Document Chunking

This notebook converts structured CRMP articles into retrieval-sized text chunks. Short articles remain intact, while long articles are split with overlap to reduce context loss at chunk boundaries.

The output is `data/processed/crmp_chunks.json`, including legal metadata and source-page references for every chunk.


## 1. Import the required libraries

Load the utilities used to read, analyze, split, and serialize the article dataset.


In [1]:
from pathlib import Path
import json
import re

## 2. Define the data paths

Locate the structured article input and set the chunk dataset destination.


In [2]:
# Run this notebook from notebooks/ so the project root is its parent.
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crmp_articles.json"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crmp_chunks.json"
)

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")


Input:  c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_articles.json
Output: c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_chunks.json


## 3. Load the parsed articles

Read the article records produced by the document parsing stage.


In [3]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

articles = data["articles"]

print(f"Artigos carregados: {len(articles)}")

Artigos carregados: 1440


## 4. Measure article lengths

Calculate character and word counts for every article.


In [4]:
article_lengths = [
    {
        "article": article["article"],
        "title": article["article_title"],
        "characters": len(article["text"]),
        "words": len(article["text"].split())
    }
    for article in articles
]

## 5. Summarize the length distribution

Report basic corpus statistics to guide chunk-size selection.


In [5]:
lengths = [x["words"] for x in article_lengths]

print(f"Artigos: {len(lengths)}")
print(f"Mínimo: {min(lengths)} palavras")
print(f"Máximo: {max(lengths)} palavras")
print(f"Média: {sum(lengths) / len(lengths):.1f} palavras")

Artigos: 1440
Mínimo: 0 palavras
Máximo: 15199 palavras
Média: 132.3 palavras


## 6. Inspect the longest articles

Review documents most likely to require splitting.


In [6]:
for item in sorted(
    article_lengths,
    key=lambda x: x["words"],
    reverse=True
)[:20]:
    print(
        item["article"],
        item["words"],
        "-",
        item["title"]
    )

H/44.º 15199 - Taxas e outras receitas municipais
124.º 10201 - 1 – Vistorias e inspeções de segurança contra o risco de
13.º 2872 - A – Parque da Trindade:
G/13.º 1959 - Isenções
24.º 1366 - Condições de instalação e manutenção de tapetes ou equiparados
B-1/31.º 857 - Escassa relevância urbanística
H/22.º 847 - Espaços verdes e arvoredo urbano
D-2/3.º-A 825 - Condições gerais aplicáveis à instalação de suportes publicitários
C-2/45.º 818 - Proibições em geral
H/27.º 794 - Trânsito e estacionamento
G/19.º 738 - Isenções ou reduções em matéria de utilização do espaço público
D-3/64.º 718 - Avenças e títulos de estacionamento nos parques de estacionamento municipais
D-1/33.º-B 680 - Ocupação do espaço público por motivo de obras particulares
47.º 677 - 1 – Plantas topográficas de localização - cópias diretas da planta da Cidade:
C-2/51.º 661 - Preservação dos espaços verdes e arvoredo em operações urbanísticas
D-1/11.º 650 - Condições de instalação e manutenção de esplanadas
B-1/8.º 632 

## 7. Set the chunking parameters

Choose the maximum chunk size and overlap in words.


In [7]:
# Balance retrieval specificity with enough legal context per chunk.
MAX_WORDS = 350
OVERLAP_WORDS = 50


## 8. Build retrieval text for an article

Combine the article identifier, title, and body into the text embedded and searched later.


In [8]:
def build_article_text(article):
    parts = []

    if article.get("article"):
        parts.append(f"Artigo {article['article']}")

    if article.get("article_title"):
        parts.append(article["article_title"])

    if article.get("text"):
        parts.append(article["text"])

    return "\n".join(parts)

## 9. Define overlapping word-based splitting

Split long text into bounded windows while retaining context between adjacent chunks.


In [9]:
def split_text_by_words(
    text,
    max_words=350,
    overlap_words=50
):
    words = text.split()

    # Keep short articles intact rather than creating unnecessary fragments.
    if len(words) <= max_words:
        return [text]

    chunks = []

    start = 0

    while start < len(words):
        end = start + max_words

        chunk_words = words[start:end]

        chunks.append(
            " ".join(chunk_words)
        )

        if end >= len(words):
            break

        # Advance by less than one full chunk to preserve boundary context.
        start = end - overlap_words

    return chunks


## 10. Test splitting on the longest article

Apply the chunker to a demanding example before processing the full corpus.


In [10]:
test_article = max(
    articles,
    key=lambda x: len(x["text"].split())
)

text = build_article_text(test_article)

test_chunks = split_text_by_words(
    text,
    max_words=MAX_WORDS,
    overlap_words=OVERLAP_WORDS
)

print("Artigo:", test_article["article"])
print("Palavras:", len(text.split()))
print("Chunks:", len(test_chunks))

Artigo: H/44.º
Palavras: 15206
Chunks: 51


## 11. Inspect the test chunks

Review chunk sizes and boundary behavior for the longest article.


In [11]:
for i, chunk in enumerate(test_chunks, start=1):
    print("=" * 100)
    print(f"CHUNK {i}")
    print(f"Palavras: {len(chunk.split())}")
    print()
    print(chunk[:1000])

CHUNK 1
Palavras: 350

Artigo H/44.º Taxas e outras receitas municipais 1 – Constituem contraordenações: a) A prática de ato ou facto sem o prévio pagamento das taxas e outras receitas municipais, salvo nos casos expressamente permitidos; b) A inexatidão ou falsidade dos elementos fornecidos pelos interessados para liquidação das taxas e outras receitas municipais. c) A não prestação da informação tributária solicitada e necessária à cobrança e liquidação das taxas municipais. Fiscalização e Sancionamento de Infrações 2 – Nos casos previstos na alínea a) do número anterior, aplicam-se as coimas previstas para a falta de licenciamento, podendo haver ainda lugar à remoção da situação ilícita. 3 – A infração prevista na alínea b) do n.º 1 é punida com coima de 100 a 800 UCM para as pessoas singulares e de 1000 a 8000 UCM para as pessoas coletivas. 4 – No caso previsto na alínea c) do n.º 1, os montantes mínimo e máximo da coima são, respetivamente, de 30 a 100 UCM. Anexos A-1 Glossário An

## 12. Create structured chunks

Generate chunk records while propagating article hierarchy and source metadata.


In [12]:
def create_chunks(
    articles,
    max_words=350,
    overlap_words=50
):
    chunks = []

    for article in articles:

        full_text = build_article_text(article)

        text_chunks = split_text_by_words(
            full_text,
            max_words=max_words,
            overlap_words=overlap_words
        )

        total_chunks = len(text_chunks)

        for index, chunk_text in enumerate(
            text_chunks,
            start=1
        ):

            # Use deterministic, ordered IDs so chunks remain traceable to their article.
            chunk_id = (
                f"{article['id']}_chunk_{index:03d}"
            )

            chunks.append({
                "id": chunk_id,

                "article_id": article["id"],
                "article": article["article"],
                "article_title": article["article_title"],

                "part": article["part"],
                "title": article["title"],
                "chapter": article["chapter"],

                "page_start": article["page_start"],
                "page_end": article["page_end"],

                "chunk_index": index,
                "num_chunks": total_chunks,

                "text": chunk_text
            })

    return chunks


## 13. Chunk the complete corpus

Apply the configured strategy to every parsed article.


In [13]:
chunks = create_chunks(
    articles,
    max_words=MAX_WORDS,
    overlap_words=OVERLAP_WORDS
)

print(f"Chunks criados: {len(chunks)}")

Chunks criados: 1617


## 14. Preview generated chunks

Inspect identifiers, metadata, sizes, and text for early chunk records.


In [14]:
for chunk in chunks[:10]:
    print("=" * 100)

    print("ID:", chunk["id"])
    print("Artigo:", chunk["article"])
    print("Epígrafe:", chunk["article_title"])
    print(
        "Chunk:",
        chunk["chunk_index"],
        "/",
        chunk["num_chunks"]
    )

    print(
        "Palavras:",
        len(chunk["text"].split())
    )

    print()
    print(chunk["text"][:1000])

ID: crmp_a_1_chunk_001
Artigo: A/1.º
Epígrafe: Objeto do código
Chunk: 1 / 1
Palavras: 83

Artigo A/1.º
Objeto do código
1 – O presente código consagra as disposições regulamentares com eficácia externa em
vigor na área do Município do Porto nos seguintes domínios:
a) Urbanismo;
b) Ambiente;
c) Gestão do espaço público;
d) Intervenção municipal sobre o exercício de atividades privadas;
e) Gestão de recursos;
f) Taxas e outras receitas municipais;
g) Fiscalização e sancionamento de infrações.
2 – Esta codificação não prejudica a existência, nos domínios referidos, de disposições
regulamentares complementares ao presente código, nele devidamente referenciadas.
ID: crmp_a_2_chunk_001
Artigo: A/2.º
Epígrafe: Objeto da Parte A
Chunk: 1 / 1
Palavras: 61

Artigo A/2.º
Objeto da Parte A
A Parte A consagra:
a) No Título I, os princípios gerais inspiradores do código, que, para além dos
princípios gerais de fonte constitucional e legal, devem orientar o Município no
desenvolvimento da sua ativid

## 15. Identify split articles

Find articles whose length exceeds the configured maximum.


In [15]:
split_articles = [
    article
    for article in articles
    if len(
        split_text_by_words(
            build_article_text(article),
            max_words=MAX_WORDS,
            overlap_words=OVERLAP_WORDS
        )
    ) > 1
]

print(
    f"Artigos divididos: "
    f"{len(split_articles)}"
)

Artigos divididos: 69


## 16. Review split behavior

Display how many chunks were produced for the longest articles.


In [16]:
for article in split_articles[:20]:
    print(
        article["article"],
        "-",
        article["article_title"]
    )

B-1/2.º - Condições gerais de edificabilidade
B-1/2.º-A - Afastamentos laterais e posteriores entre edifícios principais
B-1/5.º-A - Certidão de cumprimento dos requisitos legais para a constituição de edifício em
B-1/8.º - Bairros habitacionais
B-1/19.º - Parâmetros de dimensionamento
B-1/23.º - Tapumes e vedações
B-1/31.º - Escassa relevância urbanística
C-2/2.º - Princípios gerais
C-2/6.º - Espaços verdes privados
C-2/23.º - Comunicação do prosseguimento do procedimento e medidas de salvaguarda
C-2/45.º - Proibições em geral
C-2/51.º - Preservação dos espaços verdes e arvoredo em operações urbanísticas
C-3/15.º - Normas de circulação
D-1/2.º - Procedimentos
D-1/7.º-A - Proibições aplicáveis à ocupação do espaço público
D-1/11.º - Condições de instalação e manutenção de esplanadas
D-1/23.º-F - Condições da Licença
D-1/23.º-J - Gestão do Espaço Público
D-1/33.º-B - Ocupação do espaço público por motivo de obras particulares
D-1/42.º - Caução


## 17. Validate chunk lengths

Check the minimum, maximum, and average number of words per chunk.


In [17]:
chunk_lengths = [
    len(chunk["text"].split())
    for chunk in chunks
]

print(f"Chunks: {len(chunks)}")

print(
    f"Média de palavras: "
    f"{sum(chunk_lengths) / len(chunk_lengths):.1f}"
)

print(
    f"Máximo: "
    f"{max(chunk_lengths)}"
)

print(
    f"Mínimo: "
    f"{min(chunk_lengths)}"
)

Chunks: 1617
Média de palavras: 129.9
Máximo: 350
Mínimo: 3


## 18. Validate ID uniqueness

Ensure that every generated chunk identifier is unique.


In [18]:
chunk_ids = [
    chunk["id"]
    for chunk in chunks
]

print(
    "IDs únicos:",
    len(chunk_ids) == len(set(chunk_ids))
)

IDs únicos: False


## 19. Check for empty chunks

Confirm that the chunking process did not create blank retrieval units.


In [19]:
empty_chunks = [
    chunk["id"]
    for chunk in chunks
    if not chunk["text"].strip()
]

print(
    f"Chunks vazios: "
    f"{len(empty_chunks)}"
)

Chunks vazios: 0


## 20. Build the chunk dataset

Package chunking parameters, source metadata, and structured chunks together.


In [20]:
output = {
    "document": data["document"],
    "source_file": data["source_file"],

    "chunking": {
        "strategy": "article_based_word_sliding_window",
        "max_words": MAX_WORDS,
        "overlap_words": OVERLAP_WORDS
    },

    "num_articles": len(articles),
    "num_chunks": len(chunks),

    "chunks": chunks
}

## 21. Save the chunk dataset

Write the complete chunk collection as UTF-8 JSON for embedding generation.


In [21]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,  # Keep Portuguese characters readable in the output.
        indent=2
    )

print(
    f"Ficheiro criado: "
    f"{OUTPUT_FILE}"
)


Ficheiro criado: c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_chunks.json
